# IMPORTS

In [1]:
import pandas as pd
import re
import html
from tqdm import tqdm

tqdm.pandas()

# CLEANING

In [10]:
class Preprocessor:
    """
    Bước tiền xử lý bắt buộc cho cả 3 mô hình.

    Thứ tự thực thi (theo mức độ ưu tiên từ EDA):
      1. Decode HTML entities     (29.6% mẫu bị ảnh hưởng — HIGH)
      2. Xử lý ký tự đặc biệt (\\$ -> $)
      3. Strip HTML tags còn sót  (HTML tag remnants)
      4. Xóa source tags          (data leakage risk — CRITICAL)
      5. Xóa URL fragments        (3.1% mẫu — MEDIUM)
      6. Chuẩn hóa whitespace     (99.9% mẫu có special chars)
      7. Xóa #NAME? artifacts     (20 mẫu — LOW)
    """

    # Các source tag gây data leakage (theo EDA Table 9)
    SOURCE_TAGS = [
        r"\bReuters\b", r"\bAP\b", r"\bAFP\b", r"\bCanadian\s+Press\b"
    ]

    # URL patterns
    URL_PATTERNS = [
        r"https?://\S+",          # full URL
        r"www\.\S+",              # www.xxx
        r"\S+\.(com|net|org|gov|edu|io)\S*",  # domain references
        r"http\s+www\s+\S+",      # partial http www
    ]

    def __init__(self, lowercase: bool = False):
        """
        Args:
            lowercase: Lowercase ngay trong shared step (True cho Traditional ML).
                       Với DL và BERT, để False — lowercase ở bước sau.
        """
        self.lowercase = lowercase
        self._source_pattern = re.compile(
            "|".join(self.SOURCE_TAGS), flags=re.IGNORECASE
        )
        self._url_pattern = re.compile(
            "|".join(self.URL_PATTERNS), flags=re.IGNORECASE
        )
        self._html_tag_pattern = re.compile(r"<[^>]+>")
        self._name_artifact_pattern = re.compile(r"#NAME\?")
        self._whitespace_pattern = re.compile(r"\s+")
        # Regex cho \$
        self._escaped_dollar_pattern = re.compile(r"\\\$")

    def _decode_html_entities(self, text: str) -> str:
        """
        Decode HTML entities và xử lý ký tự bị escape.
        Ví dụ: Spacey #39;s → Spacey's | \\$100 → $100
        """
        # html.unescape xử lý &amp; &lt; &gt; &#39; &quot; v.v.
        text = html.unescape(text)

        # Xử lý dạng #39; (thiếu &) còn sót lại
        text = re.sub(r"#(\d+);", lambda m: chr(int(m.group(1))), text)

        # CHỖ THÊM MỚI: Chuyển \$ thành $
        text = self._escaped_dollar_pattern.sub("$", text)

        # Xóa &lt; &gt; còn sót (HTML tag remnants)
        text = text.replace("<", " ").replace(">", " ")
        return text

    def _strip_html_tags(self, text: str) -> str:
        """Xóa HTML tags còn sót sau khi decode (lt b gt, a href, v.v.)"""
        return self._html_tag_pattern.sub(" ", text)

    def _remove_source_tags(self, text: str) -> str:
        """
        Xóa source attribution tags để tránh data leakage.
        Reuters xuất hiện 18.4% trong Business nhưng chỉ 4.9% trong Sports.
        """
        return self._source_pattern.sub("", text)

    def _remove_urls(self, text: str) -> str:
        """Xóa URL fragments (3.1% mẫu, nhiều nhất trong Business class)."""
        return self._url_pattern.sub(" ", text)

    def _normalize_whitespace(self, text: str) -> str:
        """Collapse multiple spaces, strip leading/trailing."""
        return self._whitespace_pattern.sub(" ", text).strip()

    def _remove_artifacts(self, text: str) -> str:
        """Xóa #NAME? artifacts (20 mẫu trong dataset)."""
        return self._name_artifact_pattern.sub("", text)

    def clean(self, text: str) -> str:
        """Áp dụng toàn bộ shared pipeline cho một mẫu."""
        text = self._decode_html_entities(text)
        text = self._strip_html_tags(text)
        text = self._remove_source_tags(text)
        text = self._remove_urls(text)
        text = self._remove_artifacts(text)
        text = self._normalize_whitespace(text)
        if self.lowercase:
            text = text.lower()
        return text

In [3]:
preprocessor = Preprocessor(lowercase=False)

In [4]:
df_train = pd.read_csv("/content/train_split.csv", encoding="utf-8")
df_val = pd.read_csv("/content/val_split.csv", encoding="utf-8")
df_test = pd.read_csv("/content/test_split.csv", encoding="utf-8")

In [5]:
df_train['text'] = df_train['text'].progress_apply(preprocessor.clean)
df_val['text'] = df_val['text'].progress_apply(preprocessor.clean)
df_test['text'] = df_test['text'].progress_apply(preprocessor.clean)

100%|██████████| 25520/25520 [00:01<00:00, 18550.80it/s]


In [6]:
df_train.to_csv('train_clean.csv', index=False, encoding='utf-8')
df_val.to_csv('val_clean.csv',   index=False, encoding='utf-8')
df_test.to_csv('test_clean.csv',  index=False, encoding='utf-8')
print('\n💾 Đã lưu: train_clean.csv | val_clean.csv | test_clean.csv')


💾 Đã lưu: train_clean.csv | val_clean.csv | test_clean.csv
